In [2]:
from nse_platform.context.execution_context import ExecutionContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ec = ExecutionContext.for_local_spark("explore", "2026-01-02")
spark = ec.spark

In [3]:
df_index = spark.read.parquet("../data/silver/index/nse/year=2026/month=01/day=02").filter(F.col("index_name")=="Nifty 50")
df_index_prev = spark.read.parquet("../data/silver/index/nse/year=2026/month=01/day=01").filter(F.col("index_name")=="Nifty 50")
df_equity = spark.read.parquet("../data/silver/equity/nse/bhavcopy/year=2026/month=01/day=02")

In [4]:
df_equity.show(5)
df_index.show()
df_index_prev.show()

+----------+------+----------+-------+-------+-------+-------+-------+----------+----------------+------------------+------------+------------+
|    symbol|series|trade_date|   open|   high|    low|  close|   last|prev_close|total_traded_qty|total_traded_value|total_trades|        isin|
+----------+------+----------+-------+-------+-------+-------+-------+----------+----------------+------------------+------------+------------+
| 20MICRONS|    EQ|2026-01-02|  214.2| 217.79| 209.01|  214.3| 215.57|    212.08|          206190|     4.433638578E7|        7481|INE144J01027|
|21STCENMGM|    EQ|2026-01-02|  48.09|   50.0|  48.05|  49.39|  48.17|     49.03|            7321|         353893.24|          82|INE253B01015|
|    360ONE|    EQ|2026-01-02| 1179.7| 1213.7| 1171.2| 1200.1| 1200.0|    1179.7|          343492|     4.121338023E8|       20763|INE466L01038|
| 3IINFOLTD|    EQ|2026-01-02|  16.94|   17.1|  16.67|   16.9|   16.9|     16.66|          314404|        5316051.15|        1309|INE748

In [5]:
df_equity = df_equity.withColumn("stock_return", (F.col("close")-F.col("prev_close"))/F.col("prev_close"))

In [6]:
df_index = df_index.join(df_index_prev, on="index_name").select(df_index["*"],df_index_prev["close"].alias("prev_close"))

In [7]:
df_index = df_index.withColumn("index_return", (F.col("close")-F.col("prev_close"))/F.col("prev_close"))

In [ ]:
df_equity = (
    df_equity.withColumnRenamed("open", "stock_open")
    .withColumnRenamed("high", "stock_high")
    .withColumnRenamed("low", "stock_low")
    .withColumnRenamed("close", "stock_close")
    .withColumnRenamed("last", "stock_last")
    .withColumnRenamed("prev_close", "stock_prev_close")
)

df_index= (
    df_index.withColumnRenamed("open", "index_open")
    .withColumnRenamed("high", "index_high")
    .withColumnRenamed("low", "index_low")
    .withColumnRenamed("close", "index_close")
    .withColumnRenamed("prev_close", "index_prev_close")
)

df_gold = df_equity.join(df_index, on="trade_date")


DataFrame[trade_date: date, symbol: string, series: string, stock_open: double, stock_high: double, stock_low: double, stock_close: double, stock_last: double, stock_prev_close: double, total_traded_qty: bigint, total_traded_value: double, total_trades: bigint, isin: string, stock_return: double, index_name: string, index_open: double, index_high: double, index_low: double, index_close: double, volume: bigint, turnover_in_rs_cr: double, pe: double, pb: double, div_yield: double, index_prev_close: double, index_return: double]

In [ ]:
df_gold = df_gold.withColumn("relative_return", F.col("stock_return")-F.col("index_return"))

root
 |-- trade_date: date (nullable = true)
 |-- symbol: string (nullable = true)
 |-- series: string (nullable = true)
 |-- stock_open: double (nullable = true)
 |-- stock_high: double (nullable = true)
 |-- stock_low: double (nullable = true)
 |-- stock_close: double (nullable = true)
 |-- stock_last: double (nullable = true)
 |-- stock_prev_close: double (nullable = true)
 |-- total_traded_qty: long (nullable = true)
 |-- total_traded_value: double (nullable = true)
 |-- total_trades: long (nullable = true)
 |-- isin: string (nullable = true)
 |-- stock_return: double (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_open: double (nullable = true)
 |-- index_high: double (nullable = true)
 |-- index_low: double (nullable = true)
 |-- index_close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- turnover_in_rs_cr: double (nullable = true)
 |-- pe: double (nullable = true)
 |-- pb: double (nullable = true)
 |-- div_yield: double (nullable = true)

In [8]:
df = spark.read.format("delta").load("../data/gold/nse/year=2026/month=01/day=01/")
df.select("symbol", "stock_return", "index_return", "relative_return").show(10)
print("rows:", df.count())

+----------+--------------------+------------+---------------+
|    symbol|        stock_return|index_return|relative_return|
+----------+--------------------+------------+---------------+
| 20MICRONS|-0.00324293838417069|        NULL|           NULL|
|21STCENMGM|-0.01248741188318...|        NULL|           NULL|
|    360ONE|-0.00865546218487391|        NULL|           NULL|
| 3IINFOLTD|0.003010234798314...|        NULL|           NULL|
|   3MINDIA|0.025202904741563434|        NULL|           NULL|
|    3PLAND|0.020748020748020693|        NULL|           NULL|
|    5PAISA| 0.12565523438670056|        NULL|           NULL|
|   63MOONS|0.013908205841446454|        NULL|           NULL|
|  A2ZINFRA|-0.04810583283223085|        NULL|           NULL|
|   AAATECH|-0.04991555638956646|        NULL|           NULL|
+----------+--------------------+------------+---------------+
only showing top 10 rows
rows: 2494
